## Aplicación de técnicas para balance de clases
Propuesta de Investigación
- Curso: Estadística para el Análisis Político 2
- Nombres: Estefanía Apaza (20230487) y Diego Luyo (20230934)


##### Revisión inicial de la base de datos

In [13]:
import pandas as pd

# Cargado de la base de la Escuela de Gobierno y Políticas Públicas PUCP

# Enlace de Google Sheets publicado como CSV
link_egpp = "https://docs.google.com/spreadsheets/d/e/2PACX-1vSWEVGlxzJ-y8Q5vaQYf4EAuEASkw7rvZYb8LR02zMyDICrRVnBdiOhvDl0byS-RPlr0CMXTbTM_2fo/pub?output=csv"

df = pd.read_csv(link_egpp, encoding='utf-8')

# Verificamos que cargó mostrando las primeras filas
print(f"✅ Base cargada. Filas: {len(df)}")
df.head(5)

✅ Base cargada. Filas: 25026


,id,ano,mes_id,presidente,region,provincia_id,provincia,distrito_id,distrito,sector_1,...,sub_reclamo_id,violencia_y,periodo_politico,protesta_masiva,numero_eventos_previos,actor_laboral,actor_territorial_social,actor_economico,actor_estudiantil,actor_politico_ciudadano
0,1,1980,1,Morales Bermúdez,Lima,1501,Lima,150101,Lima,Comercial,...,105,0,1,0,0,0,0,1,0,0
1,2,1980,1,Morales Bermúdez,Lima,1501,Lima,150101,Lima,Salud,...,101,0,1,0,0,1,0,0,0,0
2,3,1980,1,Morales Bermúdez,Lima,1501,Lima,150108,Chorrillos,Personas privadas de libertad,...,407,1,1,0,0,0,0,0,0,1
3,4,1980,1,Morales Bermúdez,Lima,1501,Lima,150101,Lima,Prensa,...,104,0,1,0,0,0,0,1,0,0
4,5,1980,1,Morales Bermúdez,Cusco,801,Cusco,80101,Cusco,Municipal,...,109,0,1,0,0,1,0,0,0,0


In [14]:
print("Descripción rápida de la Base de Datos")
print(f"Columnas: {len(df.columns)}")
print(f"Observaciones: {len(df)}")
print("\nConteo de la Variable Dependiente (Protesta violenta):")
print(df['violencia_y'].value_counts())
df.columns

Descripción rápida de la Base de Datos
Columnas: 34
Observaciones: 25026

Conteo de la Variable Dependiente (Protesta violenta):
violencia_y
0    21144
1     3882
Name: count, dtype: int64


Index(['id', 'ano', 'mes_id', 'presidente', 'region', 'provincia_id',
       'provincia', 'distrito_id', 'distrito', 'sector_1', 'sector_id_1',
       'actor_1', 'actor_1_id', 'sector_id_2', 'actor_2_id', 'duracion_horas',
       'numero_participantes', 'numero_detenidos', 'adversario',
       'adversario_id', 'institucion_1', 'institucion_id', 'reclamo',
       'reclamo_id', 'sub_reclamo_id', 'violencia_y', 'periodo_politico',
       'protesta_masiva', 'numero_eventos_previos', 'actor_laboral',
       'actor_territorial_social', 'actor_economico', 'actor_estudiantil',
       'actor_politico_ciudadano'],
      dtype='object')

#### Aplicación del SMOTE

In [15]:
import sys
import subprocess

# Instalamos las librerías necesarias
try:
    import imblearn
    import statsmodels
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "imbalanced-learn", "scikit-learn", "statsmodels"])
    print("✅ ¡Instalación completada con éxito!")

from imblearn.over_sampling import SMOTE
from collections import Counter
import pandas as pd
import numpy as np

In [16]:
print("==================================================================")
print("CONFIGURACIÓN Y PROCESAMIENTO DE SMOTE")

print("-" * 65)
# Creamos las dummies usando los números de nuestra variable x "periodo_politico"
df["periodo_pre90"] = (df["periodo_politico"] == 1).astype(int)
df["periodo_90_00"] = (df["periodo_politico"] == 2).astype(int)
df["periodo_01_16"] = (df["periodo_politico"] == 3).astype(int)
# El periodo 4 queda fuera como línea base para la regresión

# Variables predictoras finales que van a entrar al modelo (solo numéricas y dummies)
columnas_X = [
    # Controles de tiempo e inercia del conflicto
    "mes_id",
    "protesta_masiva",
    "numero_eventos_previos",
    "duracion_horas",
    "numero_participantes",
    "numero_detenidos",
    # Nuestras variables independientes de periodos políticos
    "periodo_pre90",
    "periodo_90_00",
    "periodo_01_16",
    # Controles de los grupos que protestan
    "actor_laboral",
    "actor_territorial_social",
    "actor_economico",
    "actor_estudiantil",
    # Omitimos actor_politico_ciudadano para usarlo como línea base
]

# Sacamos una copia de la base para meter la limpieza antes de correr el algoritmo
df_pre_smote = df.copy()

# Reemplazamos los NAs de las variables continuas con su mediana para que no colapse el SMOTE
columnas_continuas = [
    "duracion_horas",
    "numero_participantes",
    "numero_detenidos",
    "numero_eventos_previos",
]
for col in columnas_continuas:
    if col in df_pre_smote.columns:
        df_pre_smote[col] = df_pre_smote[col].fillna(
            df_pre_smote[col].median()
        )

# Eliminamos cualquier fila que tenga la variable dependiente Y vacía para que no explote
df_pre_smote = df_pre_smote.dropna(subset=["violencia_y"])

# Separamos las matrices en X e y
X = df_pre_smote[columnas_X]
y = df_pre_smote["violencia_y"].astype(int)

print(f"Columnas cargadas para la X: {list(X.columns)}")
print(f"Distribución original de nuestra dependiente Y: {Counter(y)}")
print(
    f"  --> Proporción de la data: Por cada protesta violenta hay {Counter(y)[0]/Counter(y)[1]:.2f} pacíficas."
)
print("-" * 65)

# Corremos el SMOTE para balancear las clases
smote = SMOTE(sampling_strategy="auto", random_state=42)
X_res, y_res = smote.fit_resample(X, y)

print(f"Nueva distribución balanceada con SMOTE: {Counter(y_res)}")
print("-" * 65)

# Armamos el nuevo dataframe ya con las clases equilibradas
df_balanceado = pd.DataFrame(X_res, columns=columnas_X)
df_balanceado["violencia_y"] = y_res

# Guardamos la base balanceada para usarla en los modelos de la propuesta
output_file = "base_balanceada_final.csv"
df_balanceado.to_csv(output_file, index=False, encoding="utf-8")

print(f"Base balanceada guardada en '{output_file}'")
print(
    f"Tamaño final de la base: {df_balanceado.shape[0]} filas y {df_balanceado.shape[1]} columnas."
)
print("==================================================================")

CONFIGURACIÓN Y PROCESAMIENTO DE SMOTE
-----------------------------------------------------------------
Columnas cargadas para la X: ['mes_id', 'protesta_masiva', 'numero_eventos_previos', 'duracion_horas', 'numero_participantes', 'numero_detenidos', 'periodo_pre90', 'periodo_90_00', 'periodo_01_16', 'actor_laboral', 'actor_territorial_social', 'actor_economico', 'actor_estudiantil']
Distribución original de nuestra dependiente Y: Counter({0: 21144, 1: 3882})
  --> Proporción de la data: Por cada protesta violenta hay 5.45 pacíficas.
-----------------------------------------------------------------
Nueva distribución balanceada con SMOTE: Counter({0: 21144, 1: 21144})
-----------------------------------------------------------------
Base balanceada guardada en 'base_balanceada_final.csv'
Tamaño final de la base: 42288 filas y 14 columnas.
